# 04b — Segmentación con DeepLabV3+
**Tarea:** Segmentación semántica de lesiones en mamografías (máscara binaria: lesión vs fondo)  
**Arquitectura:** DeepLabV3+ con encoder ResNet50 preentrenado en ImageNet (Transfer Learning)  
**Estrategia:** K-Fold (k=5) sobre Train+Val → Fine-tuning en dos fases → Evaluación final en Test

## 1. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

import cv2
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import jaccard_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

## 2. Rutas y parámetros

In [ ]:
DATA_DIR    = Path('../data')
MODELS_DIR  = Path('../models')
RESULTS_DIR = Path('../results')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE       = 512        # coincide con el preprocesamiento
BATCH_SIZE     = 8          # DeepLabV3+ es pesado en memoria
LR_HEAD        = 1e-3       # fase 1: solo cabeza ASPP + decoder
LR_FULL        = 1e-4       # fase 2: fine-tuning completo
EPOCHS_PHASE1  = 10
EPOCHS_PHASE2  = 20
PATIENCE       = 5
N_FOLDS        = 5

IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

CLASS_NAMES    = ['NORM', 'Benigno', 'Maligno']

## 3. Carga de datos desde el preprocesamiento

Recuperamos los IDs generados en `01_preprocesamiento.ipynb`.  
Para segmentación usamos **imágenes + máscaras binarias** (lesión=1, fondo=0).  
Los casos `NORM` (etiqueta 0) no tienen lesión: su máscara es completamente negra.

Separamos el **test set** (intocable) del **pool train+val** sobre el que haremos K-Fold.

In [ ]:
def build_dataframe(split: str) -> pd.DataFrame:
    """
    Construye DataFrame {image_id, img_path, mask_path, label} para un split dado.
    Las máscaras binarias (.png) deben estar en DATA_DIR/split/masks/.
    Si no existe máscara para un caso NORM, se genera una máscara negra en tiempo real.
    """
    images_dir = DATA_DIR / split / 'images'
    masks_dir  = DATA_DIR / split / 'masks'
    labels_dir = DATA_DIR / split / 'labels'
    records = []
    for img_path in sorted(images_dir.glob('*.png')):
        img_id     = img_path.stem
        mask_path  = masks_dir / f'{img_id}_mask.png'
        label_file = labels_dir / f'{img_id}_label.npy'
        label      = int(np.load(label_file)) if label_file.exists() else 0
        records.append({
            'image_id':  img_id,
            'img_path':  str(img_path),
            'mask_path': str(mask_path) if mask_path.exists() else None,
            'label':     label
        })
    return pd.DataFrame(records)


df_train = build_dataframe('train')
df_val   = build_dataframe('val')
df_test  = build_dataframe('test')

# Pool train+val para K-Fold
df_trainval = pd.concat([df_train, df_val], ignore_index=True)

print(f'Train+Val: {len(df_trainval)} | Test (fijo): {len(df_test)}')
print('\nDistribución Train+Val:')
print(df_trainval['label'].value_counts().rename(index=dict(enumerate(CLASS_NAMES))))
print('\nDistribución Test:')
print(df_test['label'].value_counts().rename(index=dict(enumerate(CLASS_NAMES))))
print(f"\nCasos con máscara — Train+Val: {df_trainval['mask_path'].notna().sum()} | Test: {df_test['mask_path'].notna().sum()}")

## 4. Dataset y Transforms

El augmentation geométrico se aplica **simultáneamente** a imagen y máscara  
para mantener la correspondencia espacial.

In [ ]:
class SegmentationDataset(Dataset):
    """
    Dataset de mamografías para segmentación binaria (lesión vs fondo).
    - Imagen: escala de grises → 3 canales → normalización ImageNet.
    - Máscara: binaria (0/1), sin normalización.
    - Augmentation: flip horizontal + rotación aplicados SINCRÓNICAMENTE a imagen y máscara.
    - Los casos NORM sin máscara reciben una máscara negra (todo ceros).
    """
    def __init__(self, dataframe: pd.DataFrame, augment: bool = False):
        self.df      = dataframe.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _load_image(self, path: str) -> np.ndarray:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return img

    def _load_mask(self, path) -> np.ndarray:
        if path is None:
            return np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        return (mask > 127).astype(np.uint8)  # binarización robusta

    def _augment(self, img: np.ndarray, mask: np.ndarray):
        """Augmentation geométrico sincronizado imagen + máscara."""
        if np.random.rand() > 0.5:
            img  = cv2.flip(img, 1)
            mask = cv2.flip(mask, 1)
        angle = np.random.uniform(-15, 15)
        M     = cv2.getRotationMatrix2D((IMG_SIZE // 2, IMG_SIZE // 2), angle, 1.0)
        img   = cv2.warpAffine(img,  M, (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_LINEAR)
        mask  = cv2.warpAffine(mask, M, (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_NEAREST)
        return img, mask

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = self._load_image(row['img_path'])
        mask = self._load_mask(row['mask_path'])

        if self.augment:
            img, mask = self._augment(img, mask)

        # Imagen → 3 canales → tensor normalizado
        img_rgb = np.stack([img, img, img], axis=2).astype(np.float32) / 255.0
        mean    = np.array(IMAGENET_MEAN, dtype=np.float32)
        std     = np.array(IMAGENET_STD,  dtype=np.float32)
        img_rgb = (img_rgb - mean) / std
        img_t   = torch.from_numpy(img_rgb.transpose(2, 0, 1))  # (3, H, W)

        # Máscara → tensor float (1, H, W) para BCEWithLogitsLoss
        mask_t = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0)  # (1, H, W)

        return img_t, mask_t


def make_loaders(df_tr: pd.DataFrame, df_vl: pd.DataFrame):
    """Crea DataLoaders para un par train/val."""
    tr = DataLoader(SegmentationDataset(df_tr, augment=True),
                    batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
    vl = DataLoader(SegmentationDataset(df_vl, augment=False),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    return tr, vl


# DataLoader de test — se usa solo en la evaluación final
test_loader = DataLoader(
    SegmentationDataset(df_test, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True
)

## 5. Arquitectura — DeepLabV3+ con encoder ResNet50

DeepLabV3+ introduce dos mejoras clave frente a U-Net:

- **ASPP (Atrous Spatial Pyramid Pooling):** aplica convoluciones dilatadas a distintas tasas (6, 12, 18) sobre el feature map del encoder. Captura contexto multi-escala sin reducir resolución — crucial para detectar lesiones de distintos tamaños en mamografías.
- **Decoder ligero con skip connection de bajo nivel:** combina las features de ASPP (resolución H/16) con features tempranas del encoder (H/4) antes de la interpolación final, recuperando detalles de bordes finos.

Fine-tuning en **dos fases**:
- **Fase 1:** Encoder (backbone ResNet50) congelado, solo se entrena ASPP + decoder.
- **Fase 2:** Red completa descongelada con lr bajo para adaptar al dominio mamográfico.

In [ ]:
class ASPPConv(nn.Module):
    """Convolución dilatada con BN + ReLU para ASPP."""
    def __init__(self, in_ch: int, out_ch: int, dilation: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class ASPPPooling(nn.Module):
    """Pooling global + proyección para capturar contexto de imagen completa."""
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        size = x.shape[2:]
        out  = self.block(x)
        return F.interpolate(out, size=size, mode='bilinear', align_corners=False)


class ASPP(nn.Module):
    """
    Atrous Spatial Pyramid Pooling.
    Combina: conv 1×1 + 3 conv dilatadas (r=6,12,18) + global pooling → concat → proyección.
    """
    def __init__(self, in_ch: int = 2048, out_ch: int = 256,
                 dilations: tuple = (6, 12, 18)):
        super().__init__()
        self.conv1   = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
        self.atrous6  = ASPPConv(in_ch, out_ch, dilations[0])
        self.atrous12 = ASPPConv(in_ch, out_ch, dilations[1])
        self.atrous18 = ASPPConv(in_ch, out_ch, dilations[2])
        self.pool     = ASPPPooling(in_ch, out_ch)

        self.project  = nn.Sequential(
            nn.Conv2d(out_ch * 5, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        feats = [
            self.conv1(x),
            self.atrous6(x),
            self.atrous12(x),
            self.atrous18(x),
            self.pool(x)
        ]
        return self.project(torch.cat(feats, dim=1))


class DeepLabV3Plus(nn.Module):
    """
    DeepLabV3+ para segmentación binaria de mamografías.

    Encoder: ResNet50 preentrenado — stride=16 (output_stride=16).
      · low_level_features:  capa1  →  256 canales, H/4  (skip de detalle)
      · high_level_features: capa4  → 2048 canales, H/16 (entrada ASPP)

    ASPP: captura contexto multi-escala a tasas de dilatación 6, 12, 18.

    Decoder DeepLabV3+:
      1. Proyección de low_level_features: 256 → 48 canales.
      2. Upsampling ×4 de la salida ASPP (H/16 → H/4).
      3. Concatenación + refinamiento con 2 conv 3×3.
      4. Upsampling ×4 final (H/4 → H) + cabeza 1×1.
    """
    def __init__(self, freeze_encoder: bool = True):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        # ---- Encoder ----
        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1   # low-level:  256 ch, H/4
        self.layer2 = resnet.layer2   #             512 ch, H/8
        self.layer3 = resnet.layer3   #            1024 ch, H/16
        self.layer4 = resnet.layer4   # high-level: 2048 ch, H/32 → convertimos a H/16 abajo

        # Convertir layer3 y layer4 a output_stride=16 usando dilatación
        # (en lugar de stride=2 en layer3/layer4 usamos dilation)
        self._set_dilated_resnet()

        if freeze_encoder:
            for p in self._encoder_params():
                p.requires_grad = False

        # ---- ASPP ----
        self.aspp = ASPP(in_ch=2048, out_ch=256)

        # ---- Decoder ----
        # Proyección de low-level features (256 → 48)
        self.low_proj = nn.Sequential(
            nn.Conv2d(256, 48, 1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True)
        )
        # Refinamiento tras concatenación (256+48 → 256 → 256)
        self.decoder_conv = nn.Sequential(
            nn.Conv2d(256 + 48, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        # Cabeza de segmentación binaria
        self.head = nn.Conv2d(256, 1, 1)

    def _set_dilated_resnet(self):
        """
        Convierte ResNet50 a output_stride=16:
        - layer3: stride 2→1, dilation 1→2 en las conv 3×3
        - layer4: stride 2→1, dilation 1→4 en las conv 3×3
        Así el feature map de layer4 queda a H/16 en lugar de H/32.
        """
        for layer, dilation in [(self.layer3, 2), (self.layer4, 4)]:
            for name, module in layer.named_modules():
                if isinstance(module, nn.Conv2d):
                    if module.stride == (2, 2):
                        module.stride   = (1, 1)
                        module.dilation = (dilation, dilation)
                        module.padding  = (dilation, dilation)

    def _encoder_params(self):
        """Parámetros del backbone ResNet50."""
        for module in [self.layer0, self.layer1, self.layer2, self.layer3, self.layer4]:
            yield from module.parameters()

    def forward(self, x):
        h, w = x.shape[2:]

        # Encoder
        x       = self.layer0(x)   # (B, 64,   H/4,  W/4)
        low     = self.layer1(x)   # (B, 256,  H/4,  W/4)  ← low-level skip
        x       = self.layer2(low) # (B, 512,  H/8,  W/8)
        x       = self.layer3(x)   # (B, 1024, H/16, W/16)
        high    = self.layer4(x)   # (B, 2048, H/16, W/16)

        # ASPP sobre high-level features
        aspp_out = self.aspp(high)  # (B, 256, H/16, W/16)

        # Decoder
        # 1) Upsample ASPP ×4 → H/4
        aspp_up = F.interpolate(aspp_out, size=low.shape[2:],
                                mode='bilinear', align_corners=False)  # (B, 256, H/4, W/4)
        # 2) Proyectar low-level features
        low_proj = self.low_proj(low)   # (B, 48, H/4, W/4)
        # 3) Concatenar y refinar
        dec = self.decoder_conv(torch.cat([aspp_up, low_proj], dim=1))  # (B, 256, H/4, W/4)
        # 4) Upsample final ×4 → H
        dec = F.interpolate(dec, size=(h, w), mode='bilinear', align_corners=False)
        return self.head(dec)  # (B, 1, H, W) — logits


def build_deeplab(freeze_encoder: bool = True) -> nn.Module:
    return DeepLabV3Plus(freeze_encoder=freeze_encoder)


# Comprobación rápida de dimensiones
_model = build_deeplab().to(DEVICE)
_x     = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    _out = _model(_x)
print(f'Input:  {_x.shape}  →  Output: {_out.shape}  (esperado: [2, 1, {IMG_SIZE}, {IMG_SIZE}])')
del _model, _x, _out

## 6. Función de pérdida — Dice + BCE combinadas

La pérdida combinada **Dice + BCE** es la más usada en segmentación médica:
- **BCE** (Binary Cross-Entropy) penaliza cada píxel de forma independiente.
- **Dice Loss** maximiza el solapamiento entre predicción y ground-truth, especialmente útil con clases muy desequilibradas (lesiones pequeñas en mamografías).

In [ ]:
class DiceBCELoss(nn.Module):
    """Pérdida combinada: Dice Loss + BCE con logits."""
    def __init__(self, alpha: float = 0.5, smooth: float = 1.0):
        super().__init__()
        self.alpha  = alpha
        self.smooth = smooth
        self.bce    = nn.BCEWithLogitsLoss()

    def dice_loss(self, logits, targets):
        probs  = torch.sigmoid(logits)
        inter  = (probs * targets).sum(dim=(2, 3))
        union  = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice   = (2.0 * inter + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

    def forward(self, logits, targets):
        return self.alpha * self.dice_loss(logits, targets) + \
               (1 - self.alpha) * self.bce(logits, targets)


def dice_coefficient(preds_bin: np.ndarray, targets: np.ndarray, smooth: float = 1.0) -> float:
    """Dice coefficient (métrica, no pérdida) sobre arrays binarios numpy."""
    inter = (preds_bin * targets).sum()
    return (2.0 * inter + smooth) / (preds_bin.sum() + targets.sum() + smooth)


criterion_seg = DiceBCELoss(alpha=0.5)

## 7. Funciones de entrenamiento y evaluación

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_dice = 0.0
    n_samples  = 0
    for imgs, masks in tqdm(loader, desc='  train', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            preds_bin = (torch.sigmoid(logits) > 0.5).float()
            inter = (preds_bin * masks).sum()
            dice  = (2 * inter + 1) / (preds_bin.sum() + masks.sum() + 1)
            total_dice += dice.item() * imgs.size(0)

        total_loss += loss.item() * imgs.size(0)
        n_samples  += imgs.size(0)

    return total_loss / n_samples, total_dice / n_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_masks = [], []
    n_samples = 0

    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        loss   = criterion(logits, masks)
        total_loss += loss.item() * imgs.size(0)
        n_samples  += imgs.size(0)

        preds_bin = (torch.sigmoid(logits) > 0.5).cpu().numpy().astype(np.uint8)
        masks_np  = masks.cpu().numpy().astype(np.uint8)
        all_preds.append(preds_bin)
        all_masks.append(masks_np)

    all_preds = np.concatenate(all_preds, axis=0)  # (N, 1, H, W)
    all_masks = np.concatenate(all_masks, axis=0)

    # Métricas globales
    dice = dice_coefficient(all_preds.flatten(), all_masks.flatten())
    iou  = jaccard_score(all_masks.flatten(), all_preds.flatten(), zero_division=1)

    return total_loss / n_samples, dice, iou, all_preds, all_masks


class EarlyStopping:
    def __init__(self, patience: int, path: str):
        self.patience = patience; self.path = path
        self.best_loss = np.inf;  self.counter = 0; self.stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            torch.save(model.state_dict(), self.path)
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True


def run_phase(model, train_loader, val_loader, optimizer, criterion,
              n_epochs, device, ckpt_path, patience=PATIENCE):
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    early_stop = EarlyStopping(patience=patience, path=ckpt_path)
    history    = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': []}

    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_dice              = train_one_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, vl_dice, vl_iou, *_ = evaluate(model, val_loader, criterion, device)
        scheduler.step(vl_loss)
        early_stop(vl_loss, model)
        history['train_loss'].append(tr_loss);  history['val_loss'].append(vl_loss)
        history['train_dice'].append(tr_dice);  history['val_dice'].append(vl_dice)
        print(f'  Epoch {epoch:3d}/{n_epochs} | '
              f'Train {tr_loss:.4f}/{tr_dice:.4f} | '
              f'Val {vl_loss:.4f}/{vl_dice:.4f} (IoU {vl_iou:.4f})')
        if early_stop.stop:
            print(f'  Early stopping en época {epoch}'); break

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return history

## 8. K-Fold Cross Validation (k=5) sobre Train+Val

Usamos `StratifiedKFold` (estratificado por clase de diagnóstico) para mantener la proporción de casos NORM/Benigno/Maligno en cada fold.  
Cada fold entrena en dos fases (ASPP+decoder → red completa).  
Al final promediamos métricas y seleccionamos el mejor fold para test.

In [ ]:
skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
X            = df_trainval['img_path'].values
y            = df_trainval['label'].values

fold_results   = []   # métricas de validación por fold
fold_histories = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold}/{N_FOLDS}  |  train: {len(train_idx)}  val: {len(val_idx)}')
    print(f'{"="*60}')

    df_tr = df_trainval.iloc[train_idx].reset_index(drop=True)
    df_vl = df_trainval.iloc[val_idx].reset_index(drop=True)
    train_loader, val_loader = make_loaders(df_tr, df_vl)

    # --- Fase 1: solo ASPP + decoder (encoder congelado) ---
    model = build_deeplab(freeze_encoder=True).to(DEVICE)
    head_params = [p for p in model.parameters() if p.requires_grad]
    opt1 = optim.Adam(head_params, lr=LR_HEAD)
    print('  Fase 1 — ASPP + decoder (encoder congelado)')
    h1 = run_phase(model, train_loader, val_loader, opt1, criterion_seg,
                   EPOCHS_PHASE1, DEVICE, str(MODELS_DIR / f'deeplab_fold{fold}_phase1.pt'))

    # --- Fase 2: fine-tuning completo ---
    for p in model.parameters():
        p.requires_grad = True
    opt2 = optim.Adam(model.parameters(), lr=LR_FULL, weight_decay=1e-4)
    print('  Fase 2 — fine-tuning completo')
    h2 = run_phase(model, train_loader, val_loader, opt2, criterion_seg,
                   EPOCHS_PHASE2, DEVICE, str(MODELS_DIR / f'deeplab_fold{fold}_best.pt'))

    # Evaluar fold en validación
    _, val_dice, val_iou, _, _ = evaluate(model, val_loader, criterion_seg, DEVICE)

    fold_results.append({'fold': fold, 'val_dice': val_dice, 'val_iou': val_iou})
    fold_histories.append((h1, h2))
    print(f'  Fold {fold} → Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}')

print('\nResumen K-Fold:')
df_kfold = pd.DataFrame(fold_results)
print(df_kfold.to_string(index=False))
print(f"\nMedia Val Dice: {df_kfold['val_dice'].mean():.4f} ± {df_kfold['val_dice'].std():.4f}")
print(f"Media Val IoU:  {df_kfold['val_iou'].mean():.4f} ± {df_kfold['val_iou'].std():.4f}")

## 9. Curvas de entrenamiento 

In [ ]:
fig, axes = plt.subplots(N_FOLDS, 2, figsize=(14, N_FOLDS * 3))

for fold_idx, (h1, h2) in enumerate(fold_histories):
    tr_loss = h1['train_loss'] + h2['train_loss']
    vl_loss = h1['val_loss']   + h2['val_loss']
    tr_dice = h1['train_dice'] + h2['train_dice']
    vl_dice = h1['val_dice']   + h2['val_dice']
    pb      = len(h1['train_loss'])   # boundary entre fases

    for ax, (tr, vl), ylabel in zip(
        axes[fold_idx],
        [(tr_loss, vl_loss), (tr_dice, vl_dice)],
        ['Loss (Dice+BCE)', 'Dice Coefficient']
    ):
        ax.plot(tr, label='Train', color='steelblue')
        ax.plot(vl, label='Val',   color='tomato')
        ax.axvline(pb - 1, color='gray', linestyle='--', linewidth=0.8, label='Fin fase 1')
        ax.set_title(f'Fold {fold_idx+1} — {ylabel}', fontsize=10)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Curvas de entrenamiento por fold (Segmentación DeepLabV3+)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 10. Selección del mejor fold y evaluación en Test

Cargamos el checkpoint del fold con mayor Dice de validación  
y lo evaluamos sobre el **test set fijo** por primera y única vez.

In [ ]:
best_fold = df_kfold.loc[df_kfold['val_dice'].idxmax(), 'fold']
print(f'Mejor fold: {best_fold} (Dice val = {df_kfold.loc[df_kfold["fold"]==best_fold, "val_dice"].values[0]:.4f})')

# Cargar modelo del mejor fold
best_model = build_deeplab(freeze_encoder=False).to(DEVICE)
best_model.load_state_dict(
    torch.load(MODELS_DIR / f'deeplab_fold{best_fold}_best.pt', map_location=DEVICE)
)

test_loss, test_dice, test_iou, test_preds, test_masks = evaluate(
    best_model, test_loader, criterion_seg, DEVICE
)

pixel_acc = (test_preds == test_masks).mean()

print(f'\n{"─"*45}')
print(f'  Test Loss:          {test_loss:.4f}')
print(f'  Test Dice:          {test_dice:.4f}')
print(f'  Test IoU (Jaccard): {test_iou:.4f}')
print(f'  Test Pixel Acc:     {pixel_acc:.4f}')
print(f'{"─"*45}')

## 11. Visualización de segmentaciones — overlay imagen + máscara

In [ ]:
def denormalize(tensor):
    """Desnormaliza un tensor (3, H, W) a imagen [0, 1] RGB."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


@torch.no_grad()
def visualize_segmentations(model, dataset, device, n=6):
    """
    Muestra n ejemplos: imagen original | máscara GT | máscara predicha | overlay.
    Verde = verdadero positivo | Rojo = falso positivo | Azul = falso negativo.
    """
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(n, 4, figsize=(16, n * 4))
    col_titles = ['Imagen original', 'Máscara GT', 'Máscara predicha', 'Overlay (TP/FP/FN)']

    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=11, fontweight='bold')

    for row, idx in enumerate(indices):
        img_t, mask_t = dataset[idx]
        logit = model(img_t.unsqueeze(0).to(device))
        pred  = (torch.sigmoid(logit) > 0.5).squeeze().cpu().numpy().astype(np.uint8)
        gt    = mask_t.squeeze().numpy().astype(np.uint8)
        img_np = denormalize(img_t).permute(1, 2, 0).numpy()[:, :, 0]

        # Dice por muestra
        sample_dice = dice_coefficient(pred, gt)

        # Overlay: TP=verde, FP=rojo, FN=azul
        overlay = np.stack([img_np, img_np, img_np], axis=2)
        tp = (pred == 1) & (gt == 1)
        fp = (pred == 1) & (gt == 0)
        fn = (pred == 0) & (gt == 1)
        overlay[tp] = [0.0, 0.8, 0.0]  # verde
        overlay[fp] = [0.8, 0.0, 0.0]  # rojo
        overlay[fn] = [0.0, 0.0, 0.8]  # azul

        axes[row, 0].imshow(img_np, cmap='gray')
        axes[row, 1].imshow(gt,    cmap='gray')
        axes[row, 2].imshow(pred,  cmap='gray')
        axes[row, 3].imshow(overlay)
        axes[row, 0].set_ylabel(f'Dice={sample_dice:.3f}', fontsize=9)
        for ax in axes[row]:
            ax.axis('off')

    plt.suptitle(
        'Segmentaciones DeepLabV3+ — verde=TP | rojo=FP | azul=FN',
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    plt.show()


test_dataset = SegmentationDataset(df_test, augment=False)
visualize_segmentations(best_model, test_dataset, DEVICE)

## 12. Análisis de métricas por clase diagnóstica

Calculamos Dice e IoU separadamente para NORM, Benigno y Maligno  
para detectar si el modelo funciona mejor en algún subgrupo.

In [ ]:
@torch.no_grad()
def metrics_by_class(model, df: pd.DataFrame, device):
    """Calcula Dice e IoU por clase diagnóstica."""
    results = []
    model.eval()
    for _, row in df.iterrows():
        ds   = SegmentationDataset(pd.DataFrame([row]), augment=False)
        img_t, mask_t = ds[0]
        logit = model(img_t.unsqueeze(0).to(device))
        pred  = (torch.sigmoid(logit) > 0.5).squeeze().cpu().numpy().astype(np.uint8)
        gt    = mask_t.squeeze().numpy().astype(np.uint8)
        dice  = dice_coefficient(pred, gt)
        iou   = jaccard_score(gt.flatten(), pred.flatten(), zero_division=1)
        results.append({'label': row['label'], 'dice': dice, 'iou': iou})

    df_res = pd.DataFrame(results)
    df_res['class_name'] = df_res['label'].map(dict(enumerate(CLASS_NAMES)))
    summary = df_res.groupby('class_name')[['dice', 'iou']].agg(['mean', 'std'])
    return summary


summary = metrics_by_class(best_model, df_test, DEVICE)
print('Métricas por clase diagnóstica (Test):')
print(summary.round(4))

# Gráfico de barras
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ['dice', 'iou']):
    means = summary[metric]['mean']
    stds  = summary[metric]['std']
    bars  = ax.bar(means.index, means.values, yerr=stds.values,
                   color=['steelblue', 'seagreen', 'tomato'], capsize=5, alpha=0.85)
    ax.set_ylim(0, 1.0)
    ax.set_title(f'{metric.upper()} por clase', fontsize=11)
    ax.set_ylabel(metric.capitalize()); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=9)

plt.suptitle('Métricas de segmentación por clase — Test (DeepLabV3+)', fontsize=13)
plt.tight_layout()
plt.show()

## 13. Mapa de probabilidad (heatmap de confianza)

Visualizamos el **mapa de probabilidades** `σ(logit)` (0→1) en lugar de la máscara binaria,  
para entender la confianza del modelo píxel a píxel.  
Gracias al ASPP, DeepLabV3+ captura contexto a múltiples escalas — los mapas suelen ser más suaves y menos fragmentados que los de U-Net.

In [ ]:
@torch.no_grad()
def show_probability_maps(model, dataset, device, n=4):
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(12, n * 4))
    col_titles = ['Imagen original', 'Máscara GT', 'Mapa de probabilidad']

    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=11, fontweight='bold')

    for row, idx in enumerate(indices):
        img_t, mask_t = dataset[idx]
        logit    = model(img_t.unsqueeze(0).to(device))
        prob_map = torch.sigmoid(logit).squeeze().cpu().numpy()
        gt       = mask_t.squeeze().numpy()
        img_np   = denormalize(img_t).permute(1, 2, 0).numpy()[:, :, 0]

        axes[row, 0].imshow(img_np,   cmap='gray')
        axes[row, 1].imshow(gt,       cmap='gray')
        im = axes[row, 2].imshow(prob_map, cmap='hot', vmin=0, vmax=1)
        plt.colorbar(im, ax=axes[row, 2], fraction=0.046)
        for ax in axes[row]:
            ax.axis('off')

    plt.suptitle('Mapas de probabilidad de lesión por píxel (DeepLabV3+)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


show_probability_maps(best_model, test_dataset, DEVICE)

## 14. Guardado de resultados para evaluación comparativa

In [ ]:
# Guardar máscaras predichas y ground-truth
np.save(RESULTS_DIR / 'deeplab_test_preds.npy', test_preds)  # (N, 1, H, W) binario
np.save(RESULTS_DIR / 'deeplab_test_masks.npy', test_masks)  # (N, 1, H, W) binario

# Guardar métricas por muestra en CSV
per_sample_dice = [
    dice_coefficient(test_preds[i].flatten(), test_masks[i].flatten())
    for i in range(len(test_preds))
]
per_sample_iou = [
    jaccard_score(test_masks[i].flatten(), test_preds[i].flatten(), zero_division=1)
    for i in range(len(test_preds))
]

pd.DataFrame({
    'image_id':  df_test['image_id'].values,
    'label':     df_test['label'].values,
    'dice':      per_sample_dice,
    'iou':       per_sample_iou,
}).to_csv(RESULTS_DIR / 'deeplab_predictions.csv', index=False)

# Guardar resumen K-Fold
df_kfold.to_csv(RESULTS_DIR / 'deeplab_kfold_summary.csv', index=False)

# Guardar métricas globales
pd.DataFrame([{
    'test_loss': test_loss,
    'test_dice': test_dice,
    'test_iou':  test_iou,
    'pixel_acc': pixel_acc,
}]).to_csv(RESULTS_DIR / 'deeplab_test_metrics.csv', index=False)

print('Guardado en', RESULTS_DIR)
print('  deeplab_test_preds.npy / deeplab_test_masks.npy')
print('  deeplab_predictions.csv')
print('  deeplab_kfold_summary.csv')
print('  deeplab_test_metrics.csv')